## Retail AI Assistant (Smart-Shopper AI)
1. Overview

This project implements a simulation-based Retail AI Assistant capable of performing two roles:

* **Personal Shopper (Revenue Agent)** — recommends products based on user preferences and constraints
* **Customer Support Assistant (Operations Agent)** — evaluates return eligibility using defined business policies

The system follows a **hybrid agentic architecture**, where:

* A language model is used for **intent detection and explanation**
* Deterministic Python functions (tools) handle **data retrieval and business logic**

This design ensures both flexibility (natural language understanding) and reliability (rule-based decisions).



In [ ]:
## First of all , we need the important , necessary libraries
pip install pandas 

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install openai

  Using cached openai-2.32.0-py3-none-any.whl.metadata (31 kB)
Using cached openai-2.32.0-py3-none-any.whl (1.2 MB)
   ---------------------------------------- 0.0/204.7 kB ? eta -:--:--
   -- ------------------------------------- 10.2/204.7 kB ? eta -:--:--
   ----- --------------------------------- 30.7/204.7 kB 325.1 kB/s eta 0:00:01
   ----------- --------------------------- 61.4/204.7 kB 465.5 kB/s eta 0:00:01
   ----------------- --------------------- 92.2/204.7 kB 581.0 kB/s eta 0:00:01
   -------------------------------- ----- 174.1/204.7 kB 748.1 kB/s eta 0:00:01
   -------------------------------------- 204.7/204.7 kB 777.7 kB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [29]:
import pandas as pd
from datetime import datetime
from openai import OpenAI
import json

## 2. System Architecture

The architecture is divided into three main layers:



### 2.1 Data Layer

The system uses three data sources:

* **Product Inventory CSV** — product attributes, tags, stock, pricing
* **Orders CSV** — purchase records
* **Policy File** — return and exchange rules

These are loaded into memory using Pandas and accessed exclusively through tool functions.

In [30]:
products = pd.read_csv("C:/Users/Dell/Desktop/Retail_AI_Agent/datasets/product_inventory.csv")
orders = pd.read_csv("C:/Users/Dell/Desktop/Retail_AI_Agent/datasets/orders.csv")

with open("C:/Users/Dell/Desktop/Retail_AI_Agent/datasets/policy.txt", "r") as f:
    policy_text = f.read()

print("Products:", len(products))
print("Orders:", len(orders))

Products: 100
Orders: 100


## 2.2 Tool Layer (Deterministic Functions)

The system defines four core tools:

* `search_products(filters)`
* `get_product(product_id)`
* `get_order(order_id)`
* `evaluate_return(order_id)`

Each tool:

* Operates on real CSV datasets
* Applies deterministic filtering or rule-based logic
* Returns structured data

#### Product Recommendation Logic

The `search_products` tool:

* Filters by size availability
* Applies price constraints
* Prioritizes sale items
* Uses `bestseller_score` for ranking

#### Return Evaluation Logic

The `evaluate_return` tool enforces business rules such as:

* Clearance items → not returnable
* Sale items → limited return window (store credit only)
* Vendor exceptions (e.g., exchange-only or extended window)
* Standard return window → 14 days

All decisions are computed programmatically, ensuring consistency.

In [31]:
## Creating the function to search the products 
def search_products(filters: dict):
    df = products.copy()

    if "size" in filters:
        df = df[df["sizes_available"].astype(str).str.contains(str(filters["size"]))]

    if "max_price" in filters:
        df = df[df["price"] <= filters["max_price"]]

    if filters.get("on_sale"):
        df = df[df["is_sale"] == True]

    if "tags" in filters:
        for tag in filters["tags"]:
            df = df[df["tags"].str.contains(tag, case=False, na=False)]

    df = df.sort_values(by="bestseller_score", ascending=False)

    return df.head(5).to_dict(orient="records")

In [32]:
# Creatig the function to fetch the product 
def get_product(product_id: int):
    result = products[products["product_id"] == product_id]
    if result.empty:
        return None
    return result.iloc[0].to_dict()

In [33]:
## Creating the function to get the specific order on the basis of order_id
def get_order(order_id: object):
    result = orders[orders["order_id"] == order_id]
    if result.empty:
        return None
    return result.iloc[0].to_dict()

In [34]:
## The Function , we used here to evaluate the returns 
def evaluate_return(order_id: int):
    order = get_order(order_id)
    if not order:
        return {"error": "Order not found"}

    product = get_product(order["product_id"])
    if not product:
        return {"error": "Product not found"}

    order_date = datetime.strptime(order["order_date"], "%Y-%m-%d")
    days_since = (datetime.now() - order_date).days

    # Clearance
    if product.get("is_clearance"):
        return {
            "eligible": False,
            "reason": "Clearance items are final sale"
        }

    # Sale
    if product.get("is_sale"):
        if days_since <= 7:
            return {
                "eligible": True,
                "type": "store_credit"
            }
        return {
            "eligible": False,
            "reason": "Sale return window expired"
        }

    # Vendor rules
    if product.get("vendor") == "Aurelia Couture":
        return {
            "eligible": True,
            "type": "exchange_only"
        }

    if product.get("vendor") == "Nocturne":
        if days_since <= 21:
            return {
                "eligible": True,
                "type": "refund"
            }
        return {"eligible": False}

    # Normal items
    if days_since <= 14:
        return {
            "eligible": True,
            "type": "refund"
        }

    return {
        "eligible": False,
        "reason": "Return window expired"
    }

In [35]:
def call_tool(name, args):
    print(f"\n TOOL CALLED: {name} with {args}\n")

    if name == "search_products":
        return search_products(args)
    elif name == "get_product":
        return get_product(**args)
    elif name == "get_order":
        return get_order(**args)
    elif name == "evaluate_return":
        return evaluate_return(**args)

In [36]:
client = OpenAI()

SYSTEM_PROMPT = """
You are a retail AI assistant.

You have access to tools:
- search_products
- get_product
- get_order
- evaluate_return

Rules:
- NEVER guess data
- ALWAYS use tools when needed
- If data not found → say clearly
- Explain reasoning step-by-step

Shopping:
- Consider size, stock, price, sale, bestseller_score

Support:
- Apply return policy strictly
"""

In [37]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_products",
            "description": "Search products",
            "parameters": {
                "type": "object",
                "properties": {
                    "size": {"type": "string"},
                    "max_price": {"type": "number"},
                    "on_sale": {"type": "boolean"},
                    "tags": {"type": "array", "items": {"type": "string"}}
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_order",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "integer"}
                },
                "required": ["order_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "evaluate_return",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "integer"}
                },
                "required": ["order_id"]
            }
        }
    }
]



### 2.3 Agent Layer (LLM + Orchestration)

The agent is implemented using a Groq-hosted LLM (`llama-3.1-8b-instant`) and performs:

* Intent classification (shopping vs return)
* Response generation (human-like explanations)
* Orchestration of tool calls

Since Groq does not natively support structured tool calling, a **manual orchestration layer** is implemented:

* The LLM determines intent
* Python logic selects and executes the appropriate tool
* The LLM then explains the tool output

This creates a controlled and interpretable agent pipeline.


In [11]:
pip install Groq

  Using cached groq-1.2.0-py3-none-any.whl.metadata (16 kB)
Using cached groq-1.2.0-py3-none-any.whl (142 kB)
Note: you may need to restart the kernel to use updated packages.


In [38]:
from groq import Groq

client = Groq(api_key="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")

In [39]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Hello"}]
)

print(response.choices[0].message.content)

How can I assist you today?


In [40]:
def run_agent(user_input):
    print(f"\nUser: {user_input}\n")

    # Intent Detection
    intent_prompt = f"""
Classify the user request into one of:
1. shopping
2. return

User input: {user_input}

Respond with ONLY one word: shopping OR return
"""

    intent_response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": intent_prompt}]
    )

    intent = intent_response.choices[0].message.content.strip().lower()
    print(f"-->  Detected Intent: {intent}\n")

    # RETURN FLOW (FIXED)
    if "return" in intent:
        import re

        print("🔧 TOOL CALLED: get_order\n")

        match = re.search(r'O\d+', user_input, re.IGNORECASE)

        if not match:
            return "Please provide a valid order ID."

        order_id = match.group().upper()
        print("Extracted Order ID:", order_id)

        order = get_order(order_id)

        if not order:
            return "I could not find that order. Please check the order ID."

        print(" --> TOOL CALLED: get_product\n")
        product = get_product(order["product_id"])

        print("-->  TOOL CALLED: evaluate_return\n")
        decision = evaluate_return(order_id)
         
        # Step 5 — Convert decision into clear answer

        if "error" in decision:
            return "I could not process this return request."

        eligible = decision.get("eligible", False)

        # YES / NO output
        if eligible:
            answer = "Yes, you can return this item."
        else:
            answer = "No, this item is not eligible for return."

         # Build explanation
        explanation_prompt = f"""
        User asked: {user_input}

        Order details: 
        {order}

        Product details:
        {product}

        Return decision:
        {decision}

Generate a clear explanation:
- Start with YES or NO
- Explain why based on policy rules
- Mention if it's sale, clearance, vendor exception, or time window
"""

        explanation = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": explanation_prompt}]
)

        final_text = explanation.choices[0].message.content
        return final_text

  


    # Shopping Flow
    else:
        print(" --> TOOL CALLED: search_products\n")

        filters = {
            "size": "8" if "8" in user_input else None,
            "max_price": 300 if "300" in user_input else None,
            "on_sale": "sale" in user_input.lower(),
            "tags": ["modest"] if "modest" in user_input.lower() else []
        }

        filters = {k: v for k, v in filters.items() if v}

        results = search_products(filters)

        if not results:
            return "I couldn’t find any products matching your criteria."

        return f"Recommended Products:\n{results}"
    

## Part 1 — Personal Shopper Behavior

Example input:
"I need a modest evening gown under $300 in size 8. I prefer something on sale."

The agent must:

Filter by size availability

Check stock for the requested size

Prioritize sale items

Consider bestseller_score

Explain reasoning

The response should justify why the recommendation fits the user’s constraints, not simply list products.

In [41]:
print(run_agent("I need a modest evening gown under 200 in size 6 on sale"))


User: I need a modest evening gown under 200 in size 6 on sale

-->  Detected Intent: shopping

 --> TOOL CALLED: search_products

Recommended Products:
[{'product_id': 'P0028', 'title': 'Lumiere Style 28', 'vendor': 'Lumiere', 'price': 226, 'compare_at_price': 246, 'tags': 'sparkle,modest,bridal', 'sizes_available': '12|8|14|16|4|6', 'stock_per_size': "{'12': 15, '8': 5, '14': 2, '16': 9, '4': 16, '6': 20}", 'is_sale': True, 'is_clearance': False, 'bestseller_score': 97}, {'product_id': 'P0042', 'title': 'Nocturne Style 42', 'vendor': 'Nocturne', 'price': 330, 'compare_at_price': 350, 'tags': 'flowy,lace,modest', 'sizes_available': '16|8|2|12|6|10', 'stock_per_size': "{'16': 12, '8': 7, '2': 9, '12': 18, '6': 11, '10': 15}", 'is_sale': True, 'is_clearance': False, 'bestseller_score': 96}, {'product_id': 'P0004', 'title': 'Lumiere Style 4', 'vendor': 'Lumiere', 'price': 120, 'compare_at_price': 140, 'tags': 'modest,lace,cocktail', 'sizes_available': '14|6|8', 'stock_per_size': "{'14':

In [17]:
print(run_agent("Show me casual dresses size 6 under 200"))


User: Show me casual dresses size 6 under 200

 Detected Intent: shopping

 TOOL CALLED: search_products

Based on the given criteria, here are some good product recommendations that are suitable for a casual dress size 6 under $200:

1. **Lumiere Style 62 (P0062)** - This product is a good recommendation because it meets all the criteria.
- **Size**: The dress is available in size 6.
- **Price**: The current price is $181, which is under the budget of $200.
- **Sale status**: The product is on sale, indicating that it may offer a discount.
- **Bestseller score**: The product has a bestseller score of 96, which indicates that it's a popular product among customers.

2. **Aurelia Couture Style 3 (P0003) is not a suitable match due to the high cost ($263), but Aurelia Couture Style 38 (P0038)** - The product is a good recommendation because it meets some of the criteria.
- **Price**: The current price is $163, which is under the budget of $200.
- **Sale status**: The product is on sale.

### Part 2 — Support Reasoning

Example input:
"Order 1043 — I bought this dress last week. It doesn’t fit. Can I return it?"

The agent must:

Fetch the order

Check product attributes

Apply policy rules

Decide yes or no

Explain the decision clearly

We expect rule-based reasoning rather than guessing.

In [46]:
print(run_agent("Order O0008 — I bought this dress last week. It doesn’t fit. Can I return it?"))


User: Order O0008 — I bought this dress last week. It doesn’t fit. Can I return it?

-->  Detected Intent: return

🔧 TOOL CALLED: get_order

Extracted Order ID: O0008
 --> TOOL CALLED: get_product

-->  TOOL CALLED: evaluate_return

**YES**

Based on our return policy, the return for Order O0008 is eligible. However, there is an important consideration to note. 

The return period for sale items is typically shorter than for regular items. Unfortunately, the sale return window has expired for Order O0008, which makes it ineligible for a sale refund. However, since the product is not a clearance item and does not fall under any specific vendor exceptions (Eden Atelier being a standard vendor here), the original purchase time frame is still considered. Considering the original purchase time frame, the sale return window typically allows for a specified number of days after purchase before it expires. However, since the dress purchase was just last week and we have flexible sale return p

In [43]:
print(run_agent("Order O0029 — I bought this dress last week. It doesn’t fit. Can I return it?"))


User: Order O0029 — I bought this dress last week. It doesn’t fit. Can I return it?

-->  Detected Intent: return

🔧 TOOL CALLED: get_order

Extracted Order ID: O0029
 --> TOOL CALLED: get_product

-->  TOOL CALLED: evaluate_return

**Return Decision: NO**

The item with order ID O0029 is not eligible for return. This is because it was purchased during a sale period, and the sale return window has expired.

According to our policy rules, items purchased during a sale period can only be returned within a specific time frame, which in this case has already passed. Unfortunately, this means that we cannot accept a return for this item as per our company's return policy.

**Reason:**

Reason for non-eligibility: Sale return window expired


In [28]:
print(run_agent("Order O0014 — can I return this?"))


User: Order O0014 — can I return this?

-->  Detected Intent: return

🔧 TOOL CALLED: get_order

Extracted Order ID: O0014
 --> TOOL CALLED: get_product

-->  TOOL CALLED: evaluate_return

**NO**

You are unable to return the order O0014. 

Based on the policy rules, the product 'Silk Avenue Style 12' (ID P0012) is currently on sale, and a sale return window has expired. As a result, the customer is not eligible to return the item. 

The sale return window typically allows customers to return items within a certain timeframe (e.g., 14-30 days) from the purchase date, depending on the vendor's return policy. Once this window has expired, returns are no longer accepted, and the customer is not entitled to a refund or exchange.

In this case, since the product is not classified as clearance (which might have a different return policy) and there's no mention of a vendor exception, the standard return policy applies, and the sale return window has expired. Therefore, the return request for 

## Part 3 — Agent Behavior Requirements

The system must:

Use tool calling (function calling)

Separate reasoning from data retrieval

Prevent hallucination

Refuse when order ID is not found

Refuse when product does not exist

In [44]:
print(run_agent("Order O0124 — can I return this?"))


User: Order O0124 — can I return this?

-->  Detected Intent: return

🔧 TOOL CALLED: get_order

Extracted Order ID: O0124
I could not find that order. Please check the order ID.


In [45]:
print(run_agent("Order O454 — can I return this?"))


User: Order O454 — can I return this?

-->  Detected Intent: return

🔧 TOOL CALLED: get_order

Extracted Order ID: O454
I could not find that order. Please check the order ID.


Conclusion

This system achieves a balance between:

* **Intelligence (LLM reasoning)**
* **Control (deterministic tools)**
* **Reliability (policy enforcement)**

By separating reasoning from execution, the architecture ensures that:

* Outputs are grounded in real data
* Business rules are consistently applied
* The system remains transparent and explainable

This makes the solution suitable for real-world retail AI applications.